# Matching c14_master_v08.xlsx bibliographic references to SEAD_staging tbl_biblio

Every c14 date in the Strucke dataset is cited to a source report/paper via `author`,
`title`, `publication_year`, `journal` and `place_of_publication`. `tbl_biblio` holds SEAD's
own bibliography (`title`, `year`, `authors`, `full_reference`, `doi`, `isbn`, ...). This
notebook checks how many of Strucke's references already exist in `tbl_biblio` and, where
there's no clean exact match, proposes fuzzy candidates for manual review.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

## Load c14_master_v08.xlsx

In [2]:
excel_path = "../data/c14_master_v08.xlsx"
df = pd.read_excel(excel_path)
df[['author', 'title', 'publication_year', 'journal', 'place_of_publication']].head()

,author,title,publication_year,journal,place_of_publication
0,"Åberg, Joakim",Bergsmonument och boplatser i Jörlanda,2015.0,"Bohusläns museum, Rapport 2015:14",Uddevalla
1,"Johansson, Nils",Tre boplatser i Spekerödsdalen,1995.0,"UV Väst, Rapport 1995:1",Kungsbacka
2,"Johansson, Nils",Tre boplatser i Spekerödsdalen,1995.0,"UV Väst, Rapport 1995:1",Kungsbacka
3,"Johansson, Nils",Tre boplatser i Spekerödsdalen,1995.0,"UV Väst, Rapport 1995:1",Kungsbacka
4,"Johansson, Nils",Tre boplatser i Spekerödsdalen,1995.0,"UV Väst, Rapport 1995:1",Kungsbacka


### The bibliography columns

`journal` and `place_of_publication` don't have a dedicated match in `tbl_biblio` -- there's
no `journal` column there, and `journal` here actually holds a report-series citation like
`"UV Väst, Rapport 1995:1"` rather than a traditional journal name (this is almost entirely
Swedish CRM/contract-archaeology grey literature, not journal articles). That kind of text is
more likely embedded in `tbl_biblio.full_reference` than stored in its own field, so matching
below focuses on `title`/`author`/`year`, and `journal`/`place_of_publication` are carried
along only as context for whoever reviews the output.

In [3]:
for col in ['author', 'title', 'publication_year', 'journal', 'place_of_publication']:
    print(f"{col:22s} non-null: {df[col].notna().sum():6d} / {len(df)}   unique: {df[col].nunique():5d}")

author                 non-null:  30301 / 30301   unique:  2020
title                  non-null:  30277 / 30301   unique:  4498
publication_year       non-null:  30269 / 30301   unique:    73
journal                non-null:  29833 / 30301   unique:  4312
place_of_publication   non-null:  29605 / 30301   unique:   140


## Connect to sead_staging database

In [4]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Load `tbl_biblio`

In [5]:
biblio = pd.read_sql(
    "select biblio_id, title, year, authors, full_reference, doi, isbn from public.tbl_biblio",
    engine,
)
print(f"{len(biblio)} references in sead_staging, "
      f"{biblio['authors'].notna().sum()} with authors, {biblio['title'].notna().sum()} with a title")
biblio.head()

8943 references in sead_staging, 8542 with authors, 8903 with a title


,biblio_id,title,year,authors,full_reference,doi,isbn
0,1,"Makrofossilanalys och Pollenanalys. Fornl. 323, Värö sn, Halland.",1996,NaN,,NaN,NaN
1,2,"Toråsskolan, Halland. Makrofossilanalys av två jordprover.",2000,NaN,,NaN,NaN
2,3,"Raä 327, Onsala, Halland, Miljöarkeologisk undersökning med arkeobotanisk oc...",2000,NaN,,NaN,NaN
3,4,"Miljöarkeologiska markundersökningar inom VKB-projektet. Område 1, Stafsinge...",2000,NaN,,NaN,NaN
4,5,"Miljöarkeologiska markundersökningar inom VKB-projektet. Område 2, Stafsinge...",2000,NaN,,NaN,NaN


## Building the unique Strucke reference table

The same report is cited on dozens (sometimes hundreds) of c14-dated rows, so the actual
grain of a "reference" is the unique `(title, author, publication_year)` triple, not the raw
30,301 rows.

In [6]:
refs = (
    df.groupby(['title', 'author', 'publication_year'], dropna=False)
    .agg(n_rows=('title', 'size'),
         journal=('journal', 'first'),
         place_of_publication=('place_of_publication', 'first'))
    .reset_index()
)
print(f'{len(refs)} unique Strucke references, covering {refs["n_rows"].sum()} rows')
refs.sort_values('n_rows', ascending=False).head()

4565 unique Strucke references, covering 30301 rows


,title,author,publication_year,n_rows,journal,place_of_publication
1117,C-14 Dateringar,Protokoll,0.0,484,Stencil,NaN
2454,Järnhantering på boplatser i Halland under äldre järnålder.,"Strömberg, Bo",1991.0,236,Nya bidrag till Hallands äldsta historia nr 4,Kungsbacka
3028,Norra Spånga. Bebyggelse och samhälle under järnåldern,"Biuw, Anita",1992.0,149,Stockholmsmonografier vol 76,Borås
4536,"Östra Odarslöv 13:5, ESS-området Forntid möter framtid - volym 1–3. Skåne, L...","Brink, Kristian & Larsson, Stefan",2017.0,148,"Arkeologerna, Rapport 2017:11",Lund
1170,"Det inneslutna rummet-om kultiska hägnader, fornborgar och befästa gårdar i ...","Olausson, Michael",1995.0,145,Riksantikvarieämbetet Arkeologiska undersöknngar Skrifter nr 9,Stockholm


## Normalizing text for matching

Lowercase, strip punctuation, collapse whitespace -- applied to both sides so the same
report title written with different capitalization/punctuation still lines up.

In [7]:
import re

def norm(s):
    if pd.isna(s):
        return ''
    s = str(s).lower()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

refs['title_norm'] = refs['title'].apply(norm)
refs['author_norm'] = refs['author'].apply(norm)
biblio['title_norm'] = biblio['title'].apply(norm)
biblio['author_norm'] = biblio['authors'].apply(norm)

# blank titles are meaningless as a matching key -- and would otherwise all "exact match" each
# other (every blank Strucke title vs. every blank tbl_biblio title), a trap that inflated an
# earlier pass of this analysis to a bogus 241 "exact matches" (240 of them blank-vs-blank)
refs_valid = refs[refs['title_norm'] != ''].reset_index(drop=True)
biblio_valid = biblio[biblio['title_norm'] != ''].reset_index(drop=True)
print(f'{len(refs_valid)} / {len(refs)} Strucke refs have a usable title, '
      f'{len(biblio_valid)} / {len(biblio)} tbl_biblio rows do')

4559 / 4565 Strucke refs have a usable title, 8903 / 8943 tbl_biblio rows do


## Exact-title tier

Normalized-title equality, independent of year -- Strucke's `publication_year` and
`tbl_biblio.year` disagree or are missing often enough (report year vs. excavation year, or
one side just not filled in) that requiring them to match would silently throw away real
matches. Where a normalized title matches more than one `tbl_biblio` row (SEAD has ~114
duplicated titles itself), prefer the duplicate whose year agrees with Strucke's.

In [8]:
refs_valid['year_str'] = refs_valid['publication_year'].apply(lambda y: str(int(y)) if pd.notna(y) else None)
biblio_valid['year_str'] = biblio_valid['year'].astype(str).str.extract(r'(\d{4})')[0]

exact = refs_valid.merge(
    biblio_valid[['biblio_id', 'title', 'year_str', 'title_norm']],
    on='title_norm', how='inner', suffixes=('', '_db'),
)
exact['year_match'] = exact['year_str'] == exact['year_str_db']
exact = (
    exact.sort_values('year_match', ascending=False)
    .drop_duplicates(subset=['title_norm', 'author', 'publication_year'], keep='first')
)
print(f'{len(exact)} / {len(refs_valid)} Strucke references have an exact normalized-title match '
      f'in tbl_biblio ({exact["year_match"].sum()} of those also agree on year)')

1 / 4559 Strucke references have an exact normalized-title match in tbl_biblio (1 of those also agree on year)


## Fuzzy tier for the rest

These are almost all Swedish CRM grey-literature report titles, which share a LOT of
boilerplate phrasing (`"Arkeologisk undersökning"`, `"förundersökning"`, `"socken"`,
`"kommun"`, ...). A first attempt with edit-distance-style scoring (`rapidfuzz`
`token_sort_ratio`) looked reasonable at a glance but turned out to have a serious flaw: a
handful of short, generic `tbl_biblio` titles (e.g. *"Arkeologisk undersökning vid Raä
116"*) acted as **magnets**, winning as the "best match" for 50-200+ completely unrelated
Strucke titles just by sharing common words.

TF-IDF cosine similarity over character n-grams fixes this properly: common substrings across
the whole corpus (the boilerplate) get down-weighted automatically by IDF, while the rarer,
distinguishing substrings (site names, parish names, RAÄ numbers) drive the score. Fit on the
union of both title corpora so the IDF weights reflect real corpus-wide frequency.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
vectorizer.fit(pd.concat([refs_valid['title_norm'], biblio_valid['title_norm']]))

X_refs = vectorizer.transform(refs_valid['title_norm'])
X_biblio = vectorizer.transform(biblio_valid['title_norm'])
similarity = (X_refs @ X_biblio.T).tocsr()

best_idx = np.asarray(similarity.argmax(axis=1)).ravel()
best_score = np.asarray(similarity.max(axis=1).todense()).ravel() * 100

refs_valid['match_biblio_id'] = biblio_valid['biblio_id'].values[best_idx]
refs_valid['match_title'] = biblio_valid['title'].values[best_idx]
refs_valid['match_year'] = biblio_valid['year'].values[best_idx]
refs_valid['match_authors'] = biblio_valid['authors'].values[best_idx]
refs_valid['match_author_norm'] = biblio_valid['author_norm'].values[best_idx]
refs_valid['title_score'] = best_score

### Author corroboration and a magnet check

As a second, independent signal, also score how similar the Strucke `author` string is to
the matched row's `authors` (many `tbl_biblio` rows have no `authors` at all, so a low score
here often just means missing data rather than a wrong match -- it's a corroborating hint,
not a hard filter). Separately, flag any `match_biblio_id` that keeps winning as the "best
match" for an unusually large number of different Strucke references -- a leftover magnet
that survived the switch to TF-IDF, and a sign to double-check that particular match by hand.

In [10]:
from rapidfuzz import fuzz

refs_valid['author_score'] = [
    fuzz.token_sort_ratio(a, b) if a and b else 0
    for a, b in zip(refs_valid['author_norm'], refs_valid['match_author_norm'])
]

magnet_counts = refs_valid.loc[refs_valid['title_score'] >= 55, 'match_biblio_id'].value_counts()
refs_valid['magnet_count'] = refs_valid['match_biblio_id'].map(magnet_counts).fillna(0).astype(int)
refs_valid[['title', 'match_title', 'title_score', 'author_score', 'magnet_count']].sort_values(
    'title_score', ascending=False
).head(10)

,title,match_title,title_score,author_score,magnet_count
1027,Brattbergstorpet. Arkeologisk förundersökning av Kville 730 samt utredning i...,Brattbergstorpet. Arkeologisk förundersökning av Kville 730 samt utredning i...,100.000000,0.000000,1
1876,"Förhistoriska gårdslämningar i Onsala. Halland, Kungsbacka kommun, Onsala so...","Förhistoriska gårdslämningar i Onsala. Halland, Kungsbacka kn, Onsala sn, Lu...",96.517708,0.000000,1
3754,Stadsparken i Kalmar inför byggandet av nytt konstmuseum. Fornlämning 94. Ka...,"Stadsparken i Kalmar inför byggandet av nytt konstmuseum. Fornlämning 94, Ka...",95.889390,71.111111,1
1798,"Från mesolitisk tid till järnålder- Tanum, inte bara hällristningar","Från mesolitisk tid till järnålder - Tanum, inte bara hällristningar. UV Väs...",91.065349,0.000000,1
1197,"Dyhagen, RAÄ 5, Skänninge 2:1, 3:2, Skänninge stad, Mjölby kommun, Östergötl...","Dyhagen, RAÄ 5, Skänninge 2:1, 3:2, Skänninge stad, Mjölby kommun, Östergötl...",90.614415,73.913043,8
973,Boplatslämningar vid Larslunda. Förromersk järnålder - vikingatid. Fornlämni...,Boplatslämningar vi Larslunda. Föromersk järnålder - vikingatid. Fornlämning...,90.226148,0.000000,6
560,"Arkeologisk undersökning. Järrestad i centrum. Väg 11, sträckan Östra Tommar...","Järrestad i centrum. Väg 11, sträckan Östra Tommarp–Simrishamn. Järrestads s...",89.245488,73.333333,1
1127,Citytunnelprojektet. Bunkeflo.delområde 2 och Bubkeflo bytomt,Citytunnelprojektet. Bunkeflo - delområde 2 och Bunkeflo bytomt : rapport öv...,88.689974,0.000000,2
3629,Skogsbrukslämningar längs ny riksväg 56 sträckan Stingtorpet till Tärnsjö,Skogsbrukslämningar längs ny riksväg 56 sträckan Stingtorpet till Tärnsjö. ...,88.681489,82.352941,1
1103,Bälinge mossar. Kustbor i Uppland under yngre stenålder.,"Bälinge mossar kustbor i Uppland under yngre stenåldern. Aun, 26. Uppsala",88.678281,75.862069,1


## Confidence tiers & coverage

Thresholds picked by eyeballing samples at each band (see the score bands printed below):
titles scoring 85+ are essentially always the same report (minor OCR/typo/report-series-suffix
differences); 65-85 is a strong candidate worth a quick manual glance; 50-65 is a weak hint,
worth showing but not trusting; below 50 is noise for this boilerplate-heavy corpus and isn't
surfaced at all.

In [11]:
def tier(row):
    if row['is_exact']:
        return 'exact_title'
    if row['title_score'] >= 85:
        return 'fuzzy_high'
    if row['title_score'] >= 65:
        return 'fuzzy_medium'
    if row['title_score'] >= 50:
        return 'fuzzy_low'
    return 'none'

refs_valid['is_exact'] = refs_valid['title_norm'].isin(set(exact['title_norm']))
refs_valid['match_type'] = refs_valid.apply(tier, axis=1)

coverage = refs_valid['match_type'].value_counts()
print(coverage)
print()
resolvable = coverage.get('exact_title', 0) + coverage.get('fuzzy_high', 0) + coverage.get('fuzzy_medium', 0)
print(f'{resolvable} / {len(refs_valid)} references ({resolvable / len(refs_valid):.1%}) have a '
      f'trustworthy or near-trustworthy candidate; the rest need real bibliographic research, '
      f'not just better string matching')

match_type
none            4287
fuzzy_low        170
fuzzy_medium      86
fuzzy_high        15
exact_title        1
Name: count, dtype: int64

102 / 4559 references (2.2%) have a trustworthy or near-trustworthy candidate; the rest need real bibliographic research, not just better string matching


## Export for manual review

One row per unique Strucke reference, with the best `tbl_biblio` candidate (if any), both
similarity signals, the magnet flag, and a blank `manual_biblio_id` column to fill in once
reviewed.

In [12]:
review = refs_valid.merge(
    exact[['title_norm', 'author', 'publication_year']].assign(_exact=True),
    on=['title_norm', 'author', 'publication_year'], how='left',
).drop(columns='_exact')

review.loc[review['match_type'] == 'none', ['match_biblio_id', 'match_title', 'match_year', 'match_authors', 'title_score', 'author_score']] = [None, None, None, None, None, None]

review['needs_review'] = review['match_type'] != 'exact_title'
review['manual_biblio_id'] = pd.NA

notes = pd.Series('', index=review.index)
notes[review['match_type'] == 'fuzzy_high'] = 'title looks like the same report with minor OCR/typo/report-series differences -- spot-check before trusting'
notes[review['match_type'] == 'fuzzy_medium'] = 'plausible candidate -- confirm title/author/year by hand'
notes[review['match_type'] == 'fuzzy_low'] = 'weak signal only, high false-positive risk for this boilerplate-heavy corpus -- verify carefully or treat as no match'
notes[review['match_type'] == 'none'] = 'no candidate above the noise floor -- likely not yet in tbl_biblio'
notes[review['magnet_count'] > 5] = notes + ' [MAGNET: this tbl_biblio row matched as "best" for ' + review['magnet_count'].astype(str) + ' different Strucke references -- likely a generic/boilerplate title, treat with extra suspicion]'
review['notes'] = notes

review = review[[
    'title', 'author', 'publication_year', 'journal', 'place_of_publication', 'n_rows',
    'match_type', 'match_biblio_id', 'match_title', 'match_year', 'match_authors',
    'title_score', 'author_score', 'magnet_count',
    'needs_review', 'manual_biblio_id', 'notes',
]].sort_values('n_rows', ascending=False)

import os
out_dir = '../output/biblio'
os.makedirs(out_dir, exist_ok=True)
review.to_csv(f'{out_dir}/biblio_reference_matches.csv', index=False)
review.to_excel(f'{out_dir}/biblio_reference_matches.xlsx', index=False)
print(f'wrote {out_dir}/biblio_reference_matches.{{csv,xlsx}}: {len(review)} references covering '
      f'{review["n_rows"].sum()} rows, {review["needs_review"].sum()} flagged for manual review')
review.head(20)

wrote ../output/biblio/biblio_reference_matches.{csv,xlsx}: 4559 references covering 30277 rows, 4558 flagged for manual review


,title,author,publication_year,journal,place_of_publication,n_rows,match_type,match_biblio_id,match_title,match_year,match_authors,title_score,author_score,magnet_count,needs_review,manual_biblio_id,notes
1117,C-14 Dateringar,Protokoll,0.0,Stencil,NaN,484,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
2454,Järnhantering på boplatser i Halland under äldre järnålder.,"Strömberg, Bo",1991.0,Nya bidrag till Hallands äldsta historia nr 4,Kungsbacka,236,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
3028,Norra Spånga. Bebyggelse och samhälle under järnåldern,"Biuw, Anita",1992.0,Stockholmsmonografier vol 76,Borås,149,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
4536,"Östra Odarslöv 13:5, ESS-området Forntid möter framtid - volym 1–3. Skåne, L...","Brink, Kristian & Larsson, Stefan",2017.0,"Arkeologerna, Rapport 2017:11",Lund,148,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
1170,"Det inneslutna rummet-om kultiska hägnader, fornborgar och befästa gårdar i ...","Olausson, Michael",1995.0,Riksantikvarieämbetet Arkeologiska undersöknngar Skrifter nr 9,Stockholm,145,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
2243,Hus & Gård. Katalogdel.,"Göthberg, H., Kyhlberg, O & Vinberg, A",1995.0,Riksantikvarieämbetet. Arkeologiska undersökningar. Skrifter nr 13,Stockholm,131,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
1721,Fornlämningar mellan Snytberga och Kumla,"Ericsson, Alf m. fl",2000.0,"UV Mitt, Rapport 2000:24",Stockholm,126,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
1607,Excavations at Helgö VIII. The Ancient Monument,"Lamm, K., Reisborg, S., Kyhlberg, O. & Bertilsson,",1982.0,NaN,Stockholm,112,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
3022,Norje Sunnansund Boplatslämningar från tidigmesolitikum och järnålder,"Kjällquist, M., Emilsson, A. & Boethius, A.",2012.0,"Blekinge museum, Rapport 2014:10",Karlskrona,107,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
2246,Hus och Gård vid Lida Äng,"Appelgren, K., Nilsson, A., & Perming, A.",2002.0,"Uv Mitt, Rapport 2002:5",Stockholm,102,none,NaN,NaN,NaN,NaN,NaN,NaN,0,True,<NA>,no candidate above the noise floor -- likely not yet in tbl_biblio
